# importación de librerías

In [41]:
import pandas as pd
import sqlite3

# Ingesta de datos

In [42]:
archivo_excel = pd.ExcelFile("..\data\INSUMOS OPERACIONALES STOCK RELIX.xlsx")
print("Hojas disponibles:", archivo_excel.sheet_names)

Hojas disponibles: ['Insumos Operacionales 2025', 'INGRESO', 'SALIDA', 'STOCK ']


In [43]:
stock = pd.read_excel("..\data\INSUMOS OPERACIONALES STOCK RELIX.xlsx", sheet_name='STOCK ', skiprows=10)
stock.head()


,Unnamed: 0,ITEM,CODIGO SAP/OTRO,DESCRIPCION INSUMO,CLASIFICACIÓN,UNIDAD,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,1.1,11174821,DUCTO RECTO;PE100;PN20;280 MM;12 M;PL,PIPING,Unidades,330.0,151.0,4.0,477.0,50.0,NaN,NaN,NaN
1,NaN,1.2,11177716,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam",PIPING,Unidades,406.0,0.0,0.0,406.0,50.0,NaN,NaN,NaN
2,NaN,1.3,en catalogacion,"Tuberia HDPE 200MM DN, Flanges Moviles C150, con perforaciones diam 76mm",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0,NaN,NaN,NaN
3,NaN,1.4,11151921,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam ( CARRETE DN 500 PN10 CLASE 150)",PIPING,Unidades,26.0,0.0,0.0,26.0,50.0,"19 Cosapi , 7 patio 2",NaN,NaN
4,NaN,1.5,11019704,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0,NaN,NaN,NaN


# Transformación

## Columnas

In [44]:
stock = stock.drop(stock.columns[[0, 1, 11, 12, 13]], axis=1)

In [45]:
stock.head()

,CODIGO SAP/OTRO,DESCRIPCION INSUMO,CLASIFICACIÓN,UNIDAD,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
0,11174821,DUCTO RECTO;PE100;PN20;280 MM;12 M;PL,PIPING,Unidades,330.0,151.0,4.0,477.0,50.0
1,11177716,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam",PIPING,Unidades,406.0,0.0,0.0,406.0,50.0
2,en catalogacion,"Tuberia HDPE 200MM DN, Flanges Moviles C150, con perforaciones diam 76mm",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0
3,11151921,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam ( CARRETE DN 500 PN10 CLASE 150)",PIPING,Unidades,26.0,0.0,0.0,26.0,50.0
4,11019704,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0


In [46]:
stock.columns

Index(['CODIGO SAP/OTRO', 'DESCRIPCION INSUMO', 'CLASIFICACIÓN ', 'UNIDAD',
       'STOCK INICIAL', 'INGRESOS', 'SALIDAS', 'STOCK ACTUAL',
       'PUNTO DE REORDENAMIENTO'],
      dtype='object')

In [47]:
stock.columns = stock.columns.str.strip()
stock.columns

Index(['CODIGO SAP/OTRO', 'DESCRIPCION INSUMO', 'CLASIFICACIÓN', 'UNIDAD',
       'STOCK INICIAL', 'INGRESOS', 'SALIDAS', 'STOCK ACTUAL',
       'PUNTO DE REORDENAMIENTO'],
      dtype='object')

In [48]:
# Estandarización de nombre de columnas y eliminación de espacios
stock = stock.rename(columns={'CODIGO SAP/OTRO': 'CODIGO SAP', 
                                    'CLASIFICACIÓN': 'CLASIFICACION', 
                                    'DESCRIPCION INSUMO': 'DESCRIPCION DEL MATERIAL', 
                                    'UNIDAD': 'UM'
                                    })
stock.columns

Index(['CODIGO SAP', 'DESCRIPCION DEL MATERIAL', 'CLASIFICACION', 'UM',
       'STOCK INICIAL', 'INGRESOS', 'SALIDAS', 'STOCK ACTUAL',
       'PUNTO DE REORDENAMIENTO'],
      dtype='object')

## Categorias únicas por columna

In [49]:
stock['CLASIFICACION'].unique()

array(['PIPING ', 'PIEZAS ESPECIALES', 'VALVULAS', 'HUMECTACIÓN ',
       'PIEZÓMETROS ', 'IMPERMEABILIAZACIÓN ', 'OBRAS CIVILES ',
       'SERVICIOS GENERALES', 'ILUMINACION EMPALIZADA',
       'IMPERMEABILIZACIÓN LLAU-LLAU', 'TALLER BERLIAM', nan,
       'RIEGO MURO TRANQUE', 'INSTRUMENTACIÓN GEOTECNICA'], dtype=object)

In [50]:
stock[stock['CLASIFICACION'].isnull()]

,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
481,11220592,MAURO TUERCA STORZ 4 PULGADAS LINEA DE RIEGO,NaN,unidades,0.0,50.0,0.0,50.0,0.0


In [51]:
stock.loc[stock['CLASIFICACION'].isnull(), 'CLASIFICACION'] = 'PIEZAS ESPECIALIES'

In [52]:
stock['CLASIFICACION'] = stock['CLASIFICACION'].str.strip()

In [53]:
stock['CLASIFICACION'].unique()

array(['PIPING', 'PIEZAS ESPECIALES', 'VALVULAS', 'HUMECTACIÓN',
       'PIEZÓMETROS', 'IMPERMEABILIAZACIÓN', 'OBRAS CIVILES',
       'SERVICIOS GENERALES', 'ILUMINACION EMPALIZADA',
       'IMPERMEABILIZACIÓN LLAU-LLAU', 'TALLER BERLIAM',
       'PIEZAS ESPECIALIES', 'RIEGO MURO TRANQUE',
       'INSTRUMENTACIÓN GEOTECNICA'], dtype=object)

In [54]:
stock[stock['CODIGO SAP'].isnull()]

,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO


In [55]:
stock[stock['CODIGO SAP'].duplicated()]

,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
57,en catalogacion,"DUCTO RECTO Ø8""X500MM EXT.FLG SO Ø10"" ANSI #150/#300 REV.C.NAT.ESP.10MM 45±5 SH A",PIPING,Unidades,NaN,0.0,0.0,0.0,20.0
58,en catalogacion,"DUCTO RECTO Ø11""X500MM EXT.FLG SO Ø10"" ANSI #150/#300 REV.C.NAT.ESP.12MM 45±5 SH A",PIPING,Unidades,NaN,0.0,0.0,0.0,20.0
59,en catalogacion,"DUCTO RECTO Ø11""X300MM EXT.FLG SO Ø10"" ANSI #150/#300 REV.C.NAT.ESP.12MM 45±5 SH A",PIPING,Unidades,NaN,0.0,0.0,0.0,20.0
60,en catalogacion,"DUCTO RECTO Ø11""X1000MM EXT.FLG SO Ø10"" ANSI #150/#300 REV.C.NAT.ESP.12MM 45±5 SH A",PIPING,Unidades,NaN,0.0,0.0,0.0,20.0
61,en catalogacion,"DUCTO RECTO Ø10""X500MM EXT.FLG SO Ø10"" ANSI #150/#300 REV.C.NAT.ESP.12MM 45±5 SH A",PIPING,Unidades,NaN,0.0,0.0,0.0,20.0
62,en catalogacion,"DUCTO RECTO Ø10""X300MM EXT.FLG SO Ø10"" ANSI #150/#300 REV.C.NAT.ESP.12MM 45±5 SH A",PIPING,Unidades,NaN,0.0,0.0,0.0,20.0
63,en catalogacion,"DUCTO RECTO Ø10""X1000MM EXT.FLG SO Ø10"" ANSI #150/#300 REV.C.NAT.ESP.12MM 45±5 SH A",PIPING,Unidades,NaN,0.0,0.0,0.0,20.0
83,11021056,"DUCTO RECTO;10"" DIA X 500MM LG SLIP ON",PIEZAS ESPECIALES,Unidades,NaN,0.0,0.0,0.0,10.0
167,catalogar sap,"Flange Ciego 28"" ANSI C300",PIEZAS ESPECIALES,NaN,NaN,0.0,10.0,-10.0,2.0
243,catalogar sap,"ESPARRAGO;3/4"" X 9""LG ACERO 4340",VALVULAS,Unidades,NaN,0.0,0.0,0.0,1500.0


In [57]:
# Se estandarizan los códigos SAP pendientes pero quedan por catalogar
# Generar códigos únicos para evitar duplicados en PRIMARY KEY
contador = 1
mask = stock['CODIGO SAP'].isin(['en catalogacion', 'catalogar sap'])
for idx in stock[mask].index:
    stock.loc[idx, 'CODIGO SAP'] = f'POR_CATALOGAR_{contador:03d}'
    contador += 1

stock['DESCRIPCION DEL MATERIAL'] = stock['DESCRIPCION DEL MATERIAL'].str.strip()

In [58]:
stock[stock['CODIGO SAP'].duplicated()]

,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
83,11021056,"DUCTO RECTO;10"" DIA X 500MM LG SLIP ON",PIEZAS ESPECIALES,Unidades,NaN,0.0,0.0,0.0,10.0
335,44001452,Bines de supresor de polvo AGUASIN; TQ15808,SERVICIOS GENERALES,Bin,NaN,0.0,0.0,0.0,NaN
364,11184744,MANGUERAS DEPOSITACIÓN,SERVICIOS GENERALES,unidades,NaN,0.0,0.0,0.0,30.0
482,10-00010,"CONECTOR TWIST LOCK , IP68, # 12 AWG 220V/16A 2P+T ( conector para luminaria IP68)",ILUMINACION EMPALIZADA,unidades,0.0,400.0,0.0,400.0,0.0


### POR CONSIDERAR!!!

In [59]:
pd.set_option('display.width', None)         # Sin límite de ancho total
pd.set_option('display.max_colwidth', None)  # Sin límite de caracteres por columna

# LAS SIGUIENTES LÍNEAS SON PARA INVESTIGAR CÓDIGOS SAP DUPLICADOS
stock[stock['CODIGO SAP'] == '10-00010']

,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
375,10-00010,"ARANDELA DE REPARTICION DIAM 1/2"", 40X40X4 , AC GALVANIZADO",ILUMINACION EMPALIZADA,unidades,0.0,400.0,0.0,400.0,0.0
482,10-00010,"CONECTOR TWIST LOCK , IP68, # 12 AWG 220V/16A 2P+T ( conector para luminaria IP68)",ILUMINACION EMPALIZADA,unidades,0.0,400.0,0.0,400.0,0.0


In [60]:
stock[stock['CODIGO SAP'] == 44001452]

,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
282,44001452,Supresor de Polvo\nAGENTE SUPRESOR;AGUASIN;TQ15808,HUMECTACIÓN,Bins,NaN,0.0,0.0,0.0,15.0
335,44001452,Bines de supresor de polvo AGUASIN; TQ15808,SERVICIOS GENERALES,Bin,NaN,0.0,0.0,0.0,NaN


In [61]:
stock[stock['CODIGO SAP'] == 11021056]

,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
5,11021056,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0
83,11021056,"DUCTO RECTO;10"" DIA X 500MM LG SLIP ON",PIEZAS ESPECIALES,Unidades,NaN,0.0,0.0,0.0,10.0


### Verificación valores nulos

In [62]:
stock.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 490 entries, 0 to 489
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CODIGO SAP                490 non-null    object 
 1   DESCRIPCION DEL MATERIAL  490 non-null    object 
 2   CLASIFICACION             490 non-null    object 
 3   UM                        486 non-null    object 
 4   STOCK INICIAL             185 non-null    float64
 5   INGRESOS                  489 non-null    float64
 6   SALIDAS                   486 non-null    float64
 7   STOCK ACTUAL              489 non-null    float64
 8   PUNTO DE REORDENAMIENTO   481 non-null    float64
dtypes: float64(5), object(4)
memory usage: 34.6+ KB


In [63]:
stock['CLASIFICACION'].unique()


array(['PIPING', 'PIEZAS ESPECIALES', 'VALVULAS', 'HUMECTACIÓN',
       'PIEZÓMETROS', 'IMPERMEABILIAZACIÓN', 'OBRAS CIVILES',
       'SERVICIOS GENERALES', 'ILUMINACION EMPALIZADA',
       'IMPERMEABILIZACIÓN LLAU-LLAU', 'TALLER BERLIAM',
       'PIEZAS ESPECIALIES', 'RIEGO MURO TRANQUE',
       'INSTRUMENTACIÓN GEOTECNICA'], dtype=object)

In [64]:
stock[stock['DESCRIPCION DEL MATERIAL'].isnull()]

,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO


In [65]:
stock[stock['CODIGO SAP'].isnull()]


,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO


In [66]:
stock[stock['UM'].isnull()]


,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
166,POR_CATALOGAR_009,"Flange Ciego diam 28"" ANSI C150",PIEZAS ESPECIALES,NaN,NaN,0.0,10.0,-10.0,2.0
167,POR_CATALOGAR_010,"Flange Ciego 28"" ANSI C300",PIEZAS ESPECIALES,NaN,NaN,0.0,10.0,-10.0,2.0
321,44002471,"POLYSEAL POLIMERO;1,05G/C3;TINETA",OBRAS CIVILES,NaN,NaN,0.0,0.0,0.0,30.0
352,11215216,BOQUILLAS DEPOSITACION 50MM POLIURETANO,SERVICIOS GENERALES,NaN,NaN,0.0,0.0,0.0,50.0


In [67]:
stock.loc[stock['UM'].isnull(), 'UM'] = 'UNIDAD'

In [68]:
stock[stock['UM'].isnull()]


,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO


In [69]:
stock[stock['STOCK INICIAL'].isnull()]


,CODIGO SAP,DESCRIPCION DEL MATERIAL,CLASIFICACION,UM,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
2,POR_CATALOGAR_001,"Tuberia HDPE 200MM DN, Flanges Moviles C150, con perforaciones diam 76mm",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0
4,11019704,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0
5,11021056,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0
6,11010236,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0
9,11214458,TUBERIA HDPE 250 MM (10 pulgadas) PN20 CLASE 300 FF,PIPING,Unidades,NaN,0.0,0.0,0.0,75.0
...,...,...,...,...,...,...,...,...,...
363,11186861,"LAMPARA SOLAR;ROJA;0,220KG;280X85X80MM",SERVICIOS GENERALES,unidades,NaN,0.0,0.0,0.0,25.0
364,11184744,MANGUERAS DEPOSITACIÓN,SERVICIOS GENERALES,unidades,NaN,0.0,0.0,0.0,30.0
365,1190167,MANGUERA AGUA TTMGTPU3IN1M,SERVICIOS GENERALES,unidades,NaN,0.0,0.0,0.0,15.0
367,22003663,"DURMIENTE 6X9""X 1ml",SERVICIOS GENERALES,unidades,NaN,0.0,0.0,0.0,20.0


In [70]:
stock[['STOCK INICIAL', 'INGRESOS', 'SALIDAS', 'STOCK ACTUAL', 'PUNTO DE REORDENAMIENTO']] = stock[['STOCK INICIAL', 'INGRESOS', 'SALIDAS', 'STOCK ACTUAL', 'PUNTO DE REORDENAMIENTO']].fillna(0)

## Estandarización unidades de medida y clasificación

In [71]:
stock['UM'].unique()

array(['Unidades', 'UNIDAD', 'kit', 'unidades', 'Bins', 'M2', 'Rollos',
       'M', 'kg', 'Mt', 'Kg', 'Metros', 'KIT', 'lts', 'mt', 'Unidad',
       'Bin', 'Rollo', 'saco 25 kg', 'Kit', 'metros'], dtype=object)

In [72]:
stock['CLASIFICACION'].unique()

array(['PIPING', 'PIEZAS ESPECIALES', 'VALVULAS', 'HUMECTACIÓN',
       'PIEZÓMETROS', 'IMPERMEABILIAZACIÓN', 'OBRAS CIVILES',
       'SERVICIOS GENERALES', 'ILUMINACION EMPALIZADA',
       'IMPERMEABILIZACIÓN LLAU-LLAU', 'TALLER BERLIAM',
       'PIEZAS ESPECIALIES', 'RIEGO MURO TRANQUE',
       'INSTRUMENTACIÓN GEOTECNICA'], dtype=object)

In [73]:
# Estandarización de categorías y unidades de medida
import sys
sys.path.append('..')
from src.data_process import estandarizar_datos

stock, valores_unicos_clasificacion, valores_unicos_um = estandarizar_datos(stock)

✅ Unidades estandarizadas
✅ Clasificación estandarizada


In [74]:
valores_unicos_clasificacion

array(['PIPING', 'PIEZAS ESPECIALES', 'VALVULAS', 'HUMECTACION',
       'PIEZOMETROS', 'IMPERMEABILIZACION', 'OBRAS CIVILES',
       'SERVICIOS GENERALES', 'ILUMINACION EMPALIZADA', 'TALLER BERLIAM',
       'RIEGO MURO TRANQUE', 'INSTRUMENTACION GEOTECNICA'], dtype=object)

In [75]:
valores_unicos_um

array(['UNIDAD', 'KIT', 'BIN', 'M2', 'ROLLO', 'M', 'KG', 'LT'],
      dtype=object)

# Carga

In [76]:
import sys
import sqlite3
sys.path.append('..')

from src.database import InventarioDatabase

# instanciamos la base de datos
db = InventarioDatabase('../src/db/inventario_lp02.db')
# Creamos la tabla insumos en la base de datos
conexion = sqlite3.connect('../src/db/inventario_lp02.db')
cursor = conexion.cursor()
cursor.execute('''
        CREATE TABLE IF NOT EXISTS stock (
            "CODIGO SAP"  TEXT NOT NULL PRIMARY KEY,
            "CLASIFICACION" TEXT NOT NULL,
            "DESCRIPCION DEL MATERIAL" TEXT NOT NULL,
            "UM" TEXT NOT NULL,
            "STOCK INICIAL" INTEGER NOT NULL,
            "INGRESOS" INTEGER NOT NULL,
            "SALIDAS" INTEGER NOT NULL,
            "STOCK ACTUAL" INTEGER NOT NULL,
            "PUNTO DE REORDENAMIENTO" INTEGER NOT NULL,
            FOREIGN KEY("CODIGO SAP") REFERENCES insumos("CODIGO SAP")
        )
    ''')
conexion.commit()
conexion.close()

In [77]:
for index, row in stock.iterrows():
    db.insert_stock(
        codigo_sap=row['CODIGO SAP'],
        clasificacion=row['CLASIFICACION'],
        descripcion=row['DESCRIPCION DEL MATERIAL'],
        um=row['UM'],
        stock_inicial=row['STOCK INICIAL'],
        ingreso=row['INGRESOS'],
        salida=row['SALIDAS'],
        stock_actual=row['STOCK ACTUAL'],
        punto_reordenamiento=row['PUNTO DE REORDENAMIENTO']
    )
db.close()

✓ stock con código SAP 11174821 insertado correctamente.
✓ stock con código SAP 11177716 insertado correctamente.
✓ stock con código SAP POR_CATALOGAR_001 insertado correctamente.
✓ stock con código SAP 11151921 insertado correctamente.
✓ stock con código SAP 11019704 insertado correctamente.
✓ stock con código SAP 11021056 insertado correctamente.
✓ stock con código SAP 11010236 insertado correctamente.
✓ stock con código SAP 11009715 insertado correctamente.
✓ stock con código SAP 11214480 insertado correctamente.
✓ stock con código SAP 11214458 insertado correctamente.
✓ stock con código SAP 11214215 insertado correctamente.
✓ stock con código SAP 11214116 insertado correctamente.
✓ stock con código SAP 11010503 insertado correctamente.
✓ stock con código SAP 11010504 insertado correctamente.
✓ stock con código SAP 11010505 insertado correctamente.
✓ stock con código SAP 11010506 insertado correctamente.
✓ stock con código SAP 11010507 insertado correctamente.
✓ stock con código SAP

✓ stock con código SAP 11188679 insertado correctamente.
✓ stock con código SAP 11188678 insertado correctamente.
✓ stock con código SAP 11188677 insertado correctamente.
✓ stock con código SAP 11188676 insertado correctamente.
✓ stock con código SAP 11151922 insertado correctamente.
✓ stock con código SAP 11182677 insertado correctamente.
✓ stock con código SAP 11010715 insertado correctamente.
✓ stock con código SAP 11188664 insertado correctamente.
✓ stock con código SAP 11188663 insertado correctamente.
✓ stock con código SAP 11010713 insertado correctamente.
✓ stock con código SAP 11220106 insertado correctamente.
✓ stock con código SAP 11217090 insertado correctamente.
✓ stock con código SAP 11216847 insertado correctamente.
✓ stock con código SAP 11215311 insertado correctamente.
✓ stock con código SAP 11009709 insertado correctamente.
✓ stock con código SAP 11009710 insertado correctamente.
✓ stock con código SAP 11188685 insertado correctamente.
✓ stock con código SAP 11214913

In [78]:
# import sqlite3
conexion = sqlite3.connect('../src/db/inventario_lp02.db')
cursor = conexion.cursor()
cursor.execute("SELECT * FROM sqlite_master")
consulta = cursor.fetchall()
# tablas = pd.read_sql_query("DROP TABLE stock", conexion)
conexion.close()
consulta

[('table',
  'sqlite_sequence',
  'sqlite_sequence',
  18,
  'CREATE TABLE sqlite_sequence(name,seq)'),
 ('table',
  'insumos',
  'insumos',
  2,
  'CREATE TABLE insumos (\n            "CODIGO SAP" TEXT PRIMARY KEY,\n            "DESCRIPCION DEL MATERIAL" TEXT,\n            "CLASIFICACION" TEXT,\n            "UM" TEXT,\n            "OBSERVACIONES" TEXT\n        )'),
 ('index', 'sqlite_autoindex_insumos_1', 'insumos', 3, None),
 ('table',
  'ingresos',
  'ingresos',
  7,
  'CREATE TABLE ingresos (\n            "ID" INTEGER PRIMARY KEY AUTOINCREMENT,\n            "CODIGO SAP" TEXT NOT NULL,\n            "CLASIFICACION" TEXT NOT NULL,\n            "DESCRIPCION DEL MATERIAL" TEXT NOT NULL,\n            "UM" TEXT NOT NULL,\n            "CANTIDAD" INTEGER NOT NULL,\n            "FECHA DE INGRESO" DATE NOT NULL,\n            "RESERVA" TEXT,\n            "GUIA DESPACHO" TEXT,\n            "OC" TEXT,\n            "USO: OPERACIONES/PROYECTO" TEXT,\n            "RECIBIDO POR" TEXT,\n            "